## HAMBURGUER RESTAURANTS WEB SCRAPPING EXERCISES

In [ ]:
import pandas as pd 
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import geopandas as gpd
#--- The followinf packages are used for geospatial analysis
from utils import * #We import our geocoding function-get_coord-from utils.py
from cred_google_Template import MY_API_KEY #The file in which we have our API key
import requests 
from tqdm import tqdm  
from bs4 import BeautifulSoup 
import json
import ast
import time 
import random

## 1. Create the dataFrames:

## Data Cleansing 1rst step: GoogleMapsScrapper

In [ ]:
#Firs we create the dataframes associated with googleMapsScrapper
bucaramanga_googleMapsScrapper = read_GoogleMapsScrapper('resultados_googleMapsScrapper/bucaramanga_hamburguesas.xlsx')
floridablanca_googleMapsScrapper = read_GoogleMapsScrapper('resultados_googleMapsScrapper/floridablanca_hamburguesas.xlsx')
giron_googleMapsScrapper = read_GoogleMapsScrapper('resultados_googleMapsScrapper/giron_hamburguesas.xlsx')
piedecuesta_googleMapsScrapper = read_GoogleMapsScrapper('resultados_googleMapsScrapper/Piedecuesta_hamburguesas.xlsx')

In [ ]:
#Let's visualize the content for every dataFrame
bucaramanga_googleMapsScrapper.tail()

In [ ]:
bucaramanga_googleMapsScrapper.info()

In [ ]:
floridablanca_googleMapsScrapper.info()

In [ ]:
giron_googleMapsScrapper.info()

In [ ]:
piedecuesta_googleMapsScrapper.info()

See what is whappening? each one of the googleMapsScrapper's dataFrames has a differente name for the 'Name' column: if we don't change this, the concatenation process will result in a wrong value

In [ ]:
#for every single one of the previous dataframes, change the name of the column with 'NAME' to 'NAME'
bucaramanga_googleMapsScrapper.rename(columns={'Name (20)':'name'}, inplace=True)
floridablanca_googleMapsScrapper.rename(columns={'Name (2)':'name'}, inplace=True)
giron_googleMapsScrapper.rename(columns={'Name (97)':'name'}, inplace=True)
piedecuesta_googleMapsScrapper.rename(columns={'Name':'name'}, inplace=True)


In [ ]:
#The 4 dataFrames have exactly the same structure: let's merge them into a single dataFrame called amb_googleMapsScrapper:
amb_googleMapsScrapper = pd.concat([bucaramanga_googleMapsScrapper, floridablanca_googleMapsScrapper, giron_googleMapsScrapper, piedecuesta_googleMapsScrapper], ignore_index=True)

In [ ]:
#Let's visualize this resulting dataFrame:
amb_googleMapsScrapper.tail()

In [ ]:
#let's delete the columns: 'User', 'Description', 'Telephone', 'email', 'Claim', 'Iframe', 'Image Url', 'ORIGINAL KW', 'CID', 'id', 'Day N.', 'Saved', 'hash' for amb_googleMapsScrapper
amb_googleMapsScrapper.drop(['User', 'Description', 'Telephone', 'email', 'Claim', 'Iframe', 'Image Url', 'ORIGINAL KW', 'CID', 'id', 'Day N.', 'Saved', 'hash'], axis=1, inplace=True)

In [ ]:
#Do not forget to reset_index on the resulting dataFrame:
amb_googleMapsScrapper.reset_index(drop=True, inplace=True)
#Now let's visualize the 5 last rows:
amb_googleMapsScrapper.tail()


In [ ]:
#Let's see what columns are left in a variable called columns_left
columns_left = amb_googleMapsScrapper.columns
print(columns_left)
#Now let's visualize the 5 last rows:
amb_googleMapsScrapper.tail()

In [ ]:
#Now let's visualize the 5 last rows:
amb_googleMapsScrapper.tail()

In [ ]:
#Now let's drop duplicates based on these 3 columns: latitude, longitude and Address
amb_googleMapsScrapper.drop_duplicates(subset=['Latitude', 'Longitude', 'Address'], inplace=True)
#let's reset the index
amb_googleMapsScrapper.reset_index(drop=True, inplace=True)
#Now let's visualize the 5 last wors
amb_googleMapsScrapper.tail()

In [ ]:
amb_googleMapsScrapper.columns

In [ ]:
#let's remove rows based on Nan values: if there is at least a Nan values in one of these columns, the row will be removed: ['Rating', 'N? Reviews', 'Stars']
amb_googleMapsScrapper.dropna(how='all', inplace=True) #We'd like to keep the majority of restaurants: only rows with all NaN values will be removed
#do not forget to reset the index
amb_googleMapsScrapper.reset_index(drop=True, inplace=True)
#Now let's visualize the 5 last rows
amb_googleMapsScrapper.tail()

In [ ]:
#Finally, we will create a column named: 'categoria_puntuacion'. We will apply the function get_score_range() from utils on the column 'Rating'
#First, we see that 'Rating' is a string: we will convert it to a float: But we'll have to replace the ',' by '.' first
amb_googleMapsScrapper['Rating'] = amb_googleMapsScrapper['Rating'].str.replace(',', '.')
#Now we convert it to a float
amb_googleMapsScrapper['Rating'] = amb_googleMapsScrapper['Rating'].astype(float)
#Now we create the column 'categoria_puntuacion'
amb_googleMapsScrapper['categoria_puntuacion'] = amb_googleMapsScrapper['Rating'].apply(get_score_range)
#Now let's visualize the 5 last rows
amb_googleMapsScrapper.tail()

In [ ]:
#Let's export these three dataFrames to csf files: they will be exported in a new folder called 'resultados_finales':
amb_googleMapsScrapper.to_csv('resultados_finales/amb_googleMapsScrapper.csv', index=False)

## Data Cleansing 2nd step: InstantDataScrapper

In [ ]:
#Firs we create the dataframes associated with googleMapsScrapper
bucaramanga_instantDataScrapper = read_instantDataScrapper('resultadosInstantDataScrapper/bucaramanga_instantDataScrapper.csv')
floridablanca_instantDataScrapper = read_instantDataScrapper('resultadosInstantDataScrapper/floridablanca_instantDataScrapper.csv')
giron_instantDataScrapper =  read_instantDataScrapper('resultadosInstantDataScrapper/giron_instantDataScrapper.csv')
piedecuesta_instantDataScrapper = read_instantDataScrapper('resultadosInstantDataScrapper/piedecuesta_instantDataScrapper.csv')

In [ ]:
#Let's visualizethe content for floridablanca_instanDataScrapper
floridablanca_instantDataScrapper.tail()

In [ ]:
#let's see the info for bucaramanga_instantDataScrapper:
bucaramanga_instantDataScrapper.info()

In [ ]:
#let's see its info, so we can find relevant information about the columns and data tyes:
floridablanca_instantDataScrapper.info()

In [ ]:
#let's see it too for giron_instantDataScrapper:
giron_instantDataScrapper.info()

In [ ]:
piedecuesta_instantDataScrapper.info()

There's also a similar problem here with the instantDataScrapper dataFrames: there are 'thrash' columns that we need to get ripped off. If we concat the dataFrames how they are right now, we're going to get a wrong result!

In [ ]:
#let's create a list with the columns that are in common between the 4 of them, and the ones that we will need!
columns2 = ['nombre_restaurante', 'tipo_restaurante', 'puntuacion',
       'cantidad_comentarios', 'precio_por_persona', 'direccion_comercial',
       'tipo_consumo', 'opciones', 'entrega', 'enlace']
#now, for every single one of the previous 4 dataFrames, use this list to fix the columns names:

bucaramanga_instantDataScrapper = bucaramanga_instantDataScrapper[columns2]
floridablanca_instantDataScrapper = floridablanca_instantDataScrapper[columns2]
giron_instantDataScrapper = giron_instantDataScrapper[columns2]
piedecuesta_instantDataScrapper = piedecuesta_instantDataScrapper[columns2]
#NOW we are erady to concatenate!

In [ ]:
#The 4 dataFrames have exactly the same structure: let's merge them into a single dataFrame called amb_instantDataScrapper:
amb_instantDataScrapper = pd.concat([bucaramanga_instantDataScrapper, floridablanca_instantDataScrapper, giron_instantDataScrapper, piedecuesta_instantDataScrapper], ignore_index=True)
#Now let's visualize the resulting dataFrame:
amb_instantDataScrapper.tail()

In [ ]:
#Let's use the info() method again to see if there's relevant infno:
amb_instantDataScrapper.info()

In [ ]:
amb_instantDataScrapper.columns

In [ ]:
#Let's see the info method again: there might be some data type issues:
amb_instantDataScrapper.info()

Ojo con esto:
- precio_por_persona
- cantidad_comentarios

son columnas que representan variables continuas. Por tanto, debemos transformarlas a un valor num?rico

In [ ]:
#Transform the columns 'precio_por_persona' and 'cantidad_comentarios' to integer type:
#amb_instantDataScrapper['precio_por_persona'] = amb_instantDataScrapper['precio_por_persona'].str.replace('$', '').str.replace(',', '').astype(int)

We can see the following error: ValueError: invalid literal for int() with base 10: '30000-50000'.Let's check which the unique values are:

In [ ]:
#Let's visualize 'precio_por_persona' unique values in a variable:
amb_instantDataScrapper['precio_por_persona'].unique()

In [ ]:
#Let's drop the column 'precio_por_persona':
amb_instantDataScrapper.drop('precio_por_persona', axis=1, inplace=True)
#Do not fotget to reset index
amb_instantDataScrapper.reset_index(drop=True, inplace=True)
#Now let's visualize the 5 last rows:
amb_instantDataScrapper.tail()

In [ ]:
#Now let's transform the column cantidad_comentarios to a numeric type: but first, we will need to eliminate any string literal that is not a number, using regular expressions:
amb_instantDataScrapper['cantidad_comentarios'] = amb_instantDataScrapper['cantidad_comentarios'].str.extract('(\d+)').astype(float)


In [ ]:
#Now let's visualize the unique values of the column 'cantidad_comentarios':
amb_instantDataScrapper['cantidad_comentarios'].unique()
#Let's visualize the 5 last rows:
amb_instantDataScrapper.tail()

In [ ]:
#And finally, we will delete rows in which at least one of this columns has a Nan value: [puntuacion, cantidad_comentarios, direccion_comercial]
#amb_instantDataScrapper.dropna(subset=['puntuacion', 'cantidad_comentarios', 'direccion_comercial'], inplace=True)
amb_instantDataScrapper.dropna(how='all', inplace=True)#Remember: we will only remove those rows with all NaN values
amb_instantDataScrapper.drop_duplicates(inplace=True)#And we'll remove duplicates based on all of the rows
#Doo not forget to reset index:
amb_instantDataScrapper.reset_index(drop=True, inplace=True)
#Now let's visualize the 5 last rows:
amb_instantDataScrapper.tail()

In [ ]:
#Additionally, we will create a new column called 'categoria_puntuacion' by applying the function get_score_range() to the column 'puntuacion':
amb_instantDataScrapper['categoria_puntuacion'] = amb_instantDataScrapper['puntuacion'].apply(get_score_range)
#Now let's visualize the 5 last rows:
amb_instantDataScrapper.tail()

Now we're at the last stage of the cleansing process: let's extract the geographic coordinates based on the address column: 'direccion_comercial'

In [ ]:
#First, we will look for the unique values of 'direccion_comercial':
amb_instantDataScrapper['direccion_comercial'].unique()

In [ ]:
# We can see that the majority of this values corresponds to authentic addresses:
# Sp, we will need to create a new columns starting from 'direccion_comercial' column, which will contain the address in a format that can be used by the geocoding function:
#First, is there are float values, we will convert them to string:
amb_instantDataScrapper['direccion_comercial'] = amb_instantDataScrapper['direccion_comercial'].astype(str)
#Second STEP WILL BE REPLACING '#' for 'No'
amb_instantDataScrapper['direccion_comercial'] = amb_instantDataScrapper['direccion_comercial'].apply(lambda x : x.replace('#', 'No'))
#now  let's replace the rows with 'Cra' and 'Kr' for 'Carrera'
amb_instantDataScrapper['direccion_comercial'] = amb_instantDataScrapper['direccion_comercial'].apply(lambda x : x.replace('Cra', 'Carrera'))
amb_instantDataScrapper['direccion_comercial'] = amb_instantDataScrapper['direccion_comercial'].apply(lambda x : x.replace('Kr', 'Carrera'))
#and we will do the same for 'Cl' and 'Cll': they will be replaced by 'Calle'
amb_instantDataScrapper['direccion_comercial'] = amb_instantDataScrapper['direccion_comercial'].apply(lambda x : x.replace('Cl', 'Calle'))
#if 'Calle' or 'Carrera' are present, they should be the first word in the address:
#amb_instantDataScrapper['direccion_comercial'] = amb_instantDataScrapper['direccion_comercial'].apply(lambda x : ' '.join([x.split()[0], x.split()[1].capitalize(), ' '.join(x.split()[2:])]))
#Now let's visualize t THE LAST 5 ROWS:
amb_instantDataScrapper.tail()



In [ ]:
#Third Step: we will create a new column called 'geocoding_address' that will contain the address in a format that can be used by the geocoding function:
amb_instantDataScrapper['geocoding_address'] = amb_instantDataScrapper.apply(lambda x : x['direccion_comercial'] + ', Bucaramanga, Santander, Colombia', axis=1)
#Now let's visualize the last 5 rows:
amb_instantDataScrapper.tail()


In [ ]:
#Fourth STEP: we will use the geocoding function to get the latitude and longitude of every address in the column 'geocoding_address':
# we will call our api_key from the cred_google_Template file
MY_API_KEY = MY_API_KEY
#lets try to get the coordinates of the first address in the column 'geocoding_address':
get_coord(amb_instantDataScrapper['geocoding_address'][0], MY_API_KEY)    

In [ ]:
#Now let's implement the following interation in order to extract out coordinates: let's call our get_coor function from utils.py
coordenadas_amb = []
tiempo = [1,2,2.5]

for i in tqdm(range(len(amb_instantDataScrapper['geocoding_address'])),colour = 'green'):

    try :
    
        R = get_coord(amb_instantDataScrapper['geocoding_address'][i],MY_API_KEY)
        geo_source = R[1],R[2]
        coordenadas_amb.append(geo_source)
    
    except :
    
        coordenadas_amb.append('Error')
    

    time.sleep(random.choices(tiempo)[0])

In [ ]:
#We have to visualize the list coordenadas_amb to see if there are any errors:
coordenadas_amb

In [ ]:
#Now let's visualize the dataframe
amb_instantDataScrapper['coordenadas'] = coordenadas_amb
#lets see the last 5 rows
amb_instantDataScrapper.tail()

In [ ]:
#Let's extract the geographic coordinates into two separate columns: coordenadas[0] will be the latitude and coordenadas[1] will be the longitude:
amb_instantDataScrapper['latitude'] = amb_instantDataScrapper['coordenadas'].apply(lambda x : x[0])
amb_instantDataScrapper['longitude'] = amb_instantDataScrapper['coordenadas'].apply(lambda x : x[1])
#Let's visualize it
amb_instantDataScrapper.tail()

In [ ]:
#Export this CSV to the folder '/resultados_finales'
amb_instantDataScrapper.to_csv('resultados_finales/amb_instantDataScrapper.csv', index=False)